In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
data = pd.read_pickle("./combined_bottle_2017_cas7_t1_x11ab_ssc.pkl")
print(type(data))
print(data)


<class 'dict'>
{'obs':          cid         lon     lat                time          z         SA  \
0       45.0 -124.735664  48.500 2017-02-19 07:07:56  -1.000000  29.283489   
1       45.0 -124.735664  48.500 2017-02-19 07:07:56  -5.600000  29.310242   
2       45.0 -124.735664  48.500 2017-02-19 07:07:56  -9.900000  29.333993   
3       45.0 -124.735664  48.500 2017-02-19 07:07:56 -19.900000  30.465824   
4       45.0 -124.735664  48.500 2017-02-19 07:07:56 -29.700001  30.628531   
...      ...         ...     ...                 ...        ...        ...   
5646  8191.0 -122.428001  47.744 2017-12-18 21:43:00 -34.300000        NaN   
5647  8192.0 -122.428001  47.744 2017-12-18 21:44:00 -24.400000        NaN   
5648  8193.0 -122.428001  47.744 2017-12-18 21:45:00 -14.400000        NaN   
5649  8194.0 -122.428001  47.744 2017-12-18 21:46:00  -1.100000        NaN   
5650  8194.0 -122.428001  47.744 2017-12-18 21:46:00  -4.100000        NaN   

            CT          DO   NO3   Chl  

In [3]:
print(list(data.keys()))

['obs', 'cas7_t1_x11ab', 'ssc']


In [4]:
print(data["obs"].columns)
print(data["cas7_t1_x11ab"].columns)
print(data["ssc"].columns)

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'cid', 'cruise', 'name', 'source', 'CT',
       'SA', 'DO', 'Chl', 'NO3', 'NH4', 'TA', 'DIC'],
      dtype='object')
Index(['time', 'lat', 'lon', 'z', 'NO3', 'silicon', 'NH4', 'DIAT', 'FLAG',
       'SA', 'CT', 'TA', 'DIC', 'DO'],
      dtype='object')


In [ ]:
# assign ssc name and cid based on lat/lon and time

# make lookup table
stn_df = data['cas7_t1_x11ab'].groupby('name', as_index=False).first()
stn_df = stn_df[['name', 'lat','lon']].copy()

ssc = data['ssc'].copy()

# round coordinates to 2 decimal places (~1 km) to allow for small mismatches
ndec = 2
stn_df['lat_r'] = stn_df['lat'].round(ndec)
stn_df['lon_r'] = stn_df['lon'].round(ndec)

ssc['lat_r'] = ssc['lat'].round(ndec)
ssc['lon_r'] = ssc['lon'].round(ndec)

ssc = ssc.merge(
    stn_df[['name', 'lat_r', 'lon_r']],
    on=['lat_r', 'lon_r'],
    how='left'
)

# check how many failed to match
n_bad = ssc['name'].isna().sum()
print(f"SSC rows with no exact lat/lon match: {n_bad} out of {len(ssc)}")

# drop helper columns if you want
ssc = ssc.drop(columns=['lat_r', 'lon_r'])

# save back
data['ssc'] = ssc

SSC rows with no exact lat/lon match: 1887 out of 5937


In [ ]:
# check method on cas7_t1_x11ab as well

# make lookup table
stn_df = data['obs'].groupby('name', as_index=False).first()
stn_df = stn_df[['name', 'lat','lon']].copy()

cas7_t1_x11ab = data['cas7_t1_x11ab'].copy()
cas7_t1_x11ab = cas7_t1_x11ab.drop(columns=['name'])

# round coordinates to 2 decimal places
ndec = 2
stn_df['lat_r'] = stn_df['lat'].round(ndec)
stn_df['lon_r'] = stn_df['lon'].round(ndec)

cas7_t1_x11ab['lat_r'] = cas7_t1_x11ab['lat'].round(ndec)
cas7_t1_x11ab['lon_r'] = cas7_t1_x11ab['lon'].round(ndec)

cas7_t1_x11ab = cas7_t1_x11ab.merge(
    stn_df[['name', 'lat_r', 'lon_r']],
    on=['lat_r', 'lon_r'],
    how='left'
)

# check how many failed to match
n_bad = cas7_t1_x11ab['name'].isna().sum()
print(f"cas7 rows with no exact lat/lon match: {n_bad} out of {len(cas7_t1_x11ab)}")

# drop helper columns if you want
cas7_t1_x11ab = cas7_t1_x11ab.drop(columns=['lat_r', 'lon_r'])

cas7 rows with no exact lat/lon match: 1887 out of 5937


In [28]:
cas7_t1_x11ab = data['cas7_t1_x11ab'].copy()
n_bad = cas7_t1_x11ab['name'].isna().sum()
print(f"rows with no exact lat/lon match: {n_bad} out of {len(cas7_t1_x11ab)}")

rows with no exact lat/lon match: 1867 out of 5651


In [ ]:
# ensure datetime
data['obs']['time'] = pd.to_datetime(data['obs']['time'])
ssc = data['ssc'].copy()
ssc['time'] = pd.to_datetime(ssc['time'])

# make obs lookup table: (name, time) -> cid
obs_lookup = data['obs'].groupby('cid', as_index=False).first()
obs_lookup = obs_lookup[['cid', 'name', 'time']]

# merge SSC with obs cid
ssc = ssc.merge(
    obs_lookup,
    on=['name', 'time'],
    how='left'
)

# check failures
n_bad = ssc['cid'].isna().sum()
print(f"SSC rows with no cid match: {n_bad} out of {len(ssc)}")

# save back
data['ssc'] = ssc

KeyError: 'name'